# derived_8.2-eval-3.3 — Parallel MoE Evaluation under SOTA 1.5 Hyperparameters

This experiment evaluates the **2-regime Mixture-of-Experts (MoE) models** and baseline models on the Washington-only `derived_8.2` split, utilizing the new **SOTA 1.5 hyperparameters** optimized during the `derived_8.2-hyperparameters-1.5` sweep. We preserve feature-set quality isolation (Spec-old, Spec-new, Global-V3, Global-c1) and include per-regime residual analysis for all 2-regime models.


# Section 1: Setup and configuration

Import libraries, fix seeds, locate the project root, probe CUDA for XGBoost, and set the parallel worker count (`XGB_PARALLEL_WORKERS`, default 6). Output artifacts land under this experiment directory.


In [1]:
import os
import sys
import random
import time
import json
import zlib
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBRegressor, XGBClassifier


def find_project_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for cand in candidates:
        if (cand / "data").exists() and (cand / "d_models").exists():
            return cand
    raise FileNotFoundError("Could not locate repo root containing 'data' and 'd_models'")


PROJECT_ROOT = find_project_root()
print(f"Project root found: {PROJECT_ROOT}")

sys_path_root = str(PROJECT_ROOT)
if sys_path_root not in sys.path:
    sys.path.append(sys_path_root)

out_dir = PROJECT_ROOT / "notebooks/experiment/derived_8.2-eval-3.3"
out_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {out_dir}")

import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
print(f"Random seed set to {SEED}")

PARALLEL_WORKERS = int(os.environ.get("XGB_PARALLEL_WORKERS", "6"))
print(f"Parallel workers: {PARALLEL_WORKERS} (override with XGB_PARALLEL_WORKERS)")

plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

try:
    dummy = xgb.XGBRegressor(n_estimators=1, device="cuda")
    dummy.fit(np.array([[1.0]]), np.array([1.0]))
    XGB_DEVICE = "cuda"
    print("XGBoost CUDA support verified and enabled.")
except Exception as e:
    XGB_DEVICE = "cpu"
    print(f"XGBoost CUDA test failed ({e}). Falling back to CPU.")

print("Setup complete. Using device:", XGB_DEVICE)


Project root found: C:\Users\pan\Documents\GitHub\MDR-Project
Output directory: C:\Users\pan\Documents\GitHub\MDR-Project\notebooks\experiment\derived_8.2-eval-3.3
Random seed set to 42
Parallel workers: 4 (override with XGB_PARALLEL_WORKERS)
XGBoost CUDA support verified and enabled.
Setup complete. Using device: cuda


# Section 2: Load data splits

Load the `derived_8.2` train / val / test CSVs, parse dates, extract month and year, and form the concatenated **trainval** matrix used for all boosters. Evaluation remains on the held-out test split.


In [2]:
TRAIN_PATH = PROJECT_ROOT / "data/splits/derived_8.2/train.csv"
VAL_PATH = PROJECT_ROOT / "data/splits/derived_8.2/val.csv"
TEST_PATH = PROJECT_ROOT / "data/splits/derived_8.2/test.csv"
TARGET_COL = "soil_moisture_5cm"
T_BINARY = 0.16

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Dataset splits loaded:")
print(f"  train: {train_df.shape}")
print(f"  val:   {val_df.shape}")
print(f"  test:  {test_df.shape}")

for df in [train_df, val_df, test_df]:
    df["date"] = pd.to_datetime(df["date"])
    df["month"] = df["date"].dt.month.astype(int)
    df["year"] = df["date"].dt.year.astype(float)

trainval_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)
print(f"  trainval (concatenated): {trainval_df.shape}")

y_trval = trainval_df[TARGET_COL].values
y_te = test_df[TARGET_COL].values
y_train_only = train_df[TARGET_COL].values
test_years = sorted(test_df["year"].unique())
print(f"Test years: {test_years}")


Dataset splits loaded:
  train: (15704, 499)
  val:   (7149, 499)
  test:  (8902, 499)
  trainval (concatenated): (22853, 500)
Test years: [np.float64(2023.0), np.float64(2024.0), np.float64(2025.0)]


# Section 3: Feature inventory

Load **previous** feature sets from `previous_features.json` (eval-3.1 / metadata copy) and **new** c1 pipeline outputs from `selected_features.json` (produced by `run_feature_selection.py`). Assert every feature exists in the dataframes.


In [3]:
prev_path = out_dir / "previous_features.json"
new_path = out_dir / "selected_features.json"

if not prev_path.exists():
    raise FileNotFoundError(f"Missing {prev_path}")
if not new_path.exists():
    raise FileNotFoundError(
        f"Missing {new_path}. Run run_feature_selection.py first."
    )

with open(prev_path, "r") as f:
    PREV = json.load(f)
with open(new_path, "r") as f:
    NEW = json.load(f)

if NEW.get("partial"):
    print("[WARNING] selected_features.json is marked partial — selection may be incomplete.")

FEATURE_SET_V3 = list(PREV["global_v3"]["features"])
FEATURE_SET_C1 = list(NEW["global_c1"]["features"])

REGIME_OLD = {
    "dry": list(PREV["binary_regime"]["dry"]["features"]),
    "wet": list(PREV["binary_regime"]["wet"]["features"]),
}
REGIME_NEW = {
    "dry": list(NEW["binary_regime"]["dry"]["features"]),
    "wet": list(NEW["binary_regime"]["wet"]["features"]),
}

CLUSTER_OLD = {
    strat: {c: list(payload["features"]) for c, payload in clusters.items()}
    for strat, clusters in PREV["clusters"].items()
}
CLUSTER_NEW = {
    strat: {c: list(payload["features"]) for c, payload in clusters.items()}
    for strat, clusters in NEW["clusters"].items()
}

print(f"Global V3: {len(FEATURE_SET_V3)} features")
print(f"Global c1: {len(FEATURE_SET_C1)} features")
print(f"Binary old dry/wet: {len(REGIME_OLD['dry'])}/{len(REGIME_OLD['wet'])}")
print(f"Binary new dry/wet: {len(REGIME_NEW['dry'])}/{len(REGIME_NEW['wet'])}")
for strat in CLUSTER_OLD:
    print(f"  Cluster old {strat}: " + ", ".join(f"c{c}={len(feats)}" for c, feats in CLUSTER_OLD[strat].items()))
for strat in CLUSTER_NEW:
    print(f"  Cluster new {strat}: " + ", ".join(f"c{c}={len(feats)}" for c, feats in CLUSTER_NEW[strat].items()))

# Column presence checks
all_needed = set(FEATURE_SET_V3) | set(FEATURE_SET_C1)
for feats in REGIME_OLD.values():
    all_needed |= set(feats)
for feats in REGIME_NEW.values():
    all_needed |= set(feats)
for strat in list(CLUSTER_OLD.values()) + list(CLUSTER_NEW.values()):
    for feats in strat.values():
        all_needed |= set(feats)

missing = sorted(c for c in all_needed if c not in trainval_df.columns)
if missing:
    raise ValueError(f"{len(missing)} features missing from trainval: {missing[:20]}")
print(f"All {len(all_needed)} referenced features present in trainval.")


Global V3: 47 features
Global c1: 50 features
Binary old dry/wet: 27/45
Binary new dry/wet: 20/50
  Cluster old Univariate_G_API_k2: c0=22, c1=1
  Cluster old Clustering_Dynamic_k2: c0=49, c1=47
  Cluster old Seasonal_Binary_k2: c0=49, c1=34
  Cluster new Univariate_G_API_k2: c0=20, c1=1
  Cluster new Clustering_Dynamic_k2: c0=50, c1=39
  Cluster new Seasonal_Binary_k2: c0=50, c1=46
All 194 referenced features present in trainval.


# Section 4: Helpers (metrics, routing, plots)

Define quantile/KMeans routers, metric computation, diagnostic plotting, yearly R² line charts, and frozen 1.3-lite hyperparameters. `n_jobs=1` avoids oversubscription when models train in a thread pool.


In [ ]:
class QuantileBinner:
    def __init__(self, K):
        self.K = K
        self.thresholds = []

    def fit(self, series):
        val = series.fillna(series.mean())
        self.thresholds = [val.quantile(i / self.K) for i in range(1, self.K)]

    def predict(self, series):
        val = series.fillna(series.mean())
        if self.K == 2:
            return np.where(val < self.thresholds[0], 0, 1)
        raise NotImplementedError("Only K=2")


class KMeansClusterer:
    def __init__(self, cols, K):
        self.cols = cols
        self.K = K
        self.means = None
        self.scaler = StandardScaler()
        self.kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)

    def fit(self, df):
        X = df[self.cols].copy()
        self.means = X.mean()
        X = X.fillna(self.means)
        self.kmeans.fit(self.scaler.fit_transform(X))

    def predict(self, df):
        X = df[self.cols].copy().fillna(self.means)
        return self.kmeans.predict(self.scaler.transform(X))


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    err = y_true - y_pred
    ae = np.abs(err)
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    ubrmse = np.sqrt(
        np.mean(((y_true - np.mean(y_true)) - (y_pred - np.mean(y_pred))) ** 2)
    )
    bias = np.mean(err)
    med_ae = np.median(ae)
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        pearson = float("nan")
    else:
        pearson = np.corrcoef(y_true, y_pred)[0, 1]
    return {
        "R2": r2,
        "RMSE": rmse,
        "ubRMSE": ubrmse,
        "Bias": bias,
        "MAE": mae,
        "Med|Err|": med_ae,
        "Pearson": pearson,
    }


def plot_diagnostics(name, y_test, pred_test, test_df, out_dir):
    y_test = np.asarray(y_test).ravel()
    pred_test = np.asarray(pred_test).ravel()
    res = y_test - pred_test
    test_years_local = sorted(test_df["year"].unique())
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    ax = axes[0, 0]
    ax.scatter(y_test, pred_test, s=8, alpha=0.5, color="#1f77b4")
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "k--", lw=1.5)
    ax.set_xlabel("True Soil Moisture")
    ax.set_ylabel("Predicted Soil Moisture")
    ax.set_title(f"Overall True vs Pred\nR2 = {r2_score(y_test, pred_test):.4f}")
    ax.grid(True, linestyle="--", alpha=0.6)

    ax = axes[1, 0]
    ax.scatter(y_test, res, s=8, alpha=0.5, color="#d62728")
    ax.axhline(0, color="k", linestyle="--", lw=1.5)
    ax.set_xlabel("True Soil Moisture")
    ax.set_ylabel("Residual (true - pred)")
    ax.set_title("Overall Residuals")
    ax.grid(True, linestyle="--", alpha=0.6)

    for idx_yr, yr in enumerate(test_years_local):
        col_idx = idx_yr + 1
        if col_idx > 3:
            break
        mask = (test_df["year"] == yr).values
        y_yr = y_test[mask]
        pred_yr = pred_test[mask]
        res_yr = res[mask]

        ax = axes[0, col_idx]
        ax.scatter(y_yr, pred_yr, s=8, alpha=0.5, color="#1f77b4")
        if len(y_yr) > 0:
            ax.plot([y_yr.min(), y_yr.max()], [y_yr.min(), y_yr.max()], "k--", lw=1.5)
        ax.set_xlabel("True Soil Moisture")
        ax.set_ylabel("Predicted Soil Moisture")
        r2_yr = r2_score(y_yr, pred_yr) if len(y_yr) > 1 else float("nan")
        ax.set_title(f"Year {int(yr)} True vs Pred\nR2 = {r2_yr:.4f}")
        ax.grid(True, linestyle="--", alpha=0.6)

        ax = axes[1, col_idx]
        ax.scatter(y_yr, res_yr, s=8, alpha=0.5, color="#d62728")
        ax.axhline(0, color="k", linestyle="--", lw=1.5)
        ax.set_xlabel("True Soil Moisture")
        ax.set_ylabel("Residual (true - pred)")
        ax.set_title(f"Year {int(yr)} Residuals")
        ax.grid(True, linestyle="--", alpha=0.6)

    plt.suptitle(f"Model Diagnostics: {name}", fontsize=16, fontweight="bold", y=0.98)
    plt.tight_layout()
    clean_name = (
        name.lower()
        .replace(" ", "_")
        .replace(":", "")
        .replace(".", "")
        .replace("=", "")
        .replace("(", "")
        .replace(")", "")
        .replace("/", "_")
    )
    fn = f"diagnostics_{clean_name}.png"
    plt.savefig(out_dir / fn, dpi=150)
    plt.close()


def plot_per_regime_diagnostics(name, y_test, pred_test, labels_te, test_df, out_dir):
    if labels_te is None:
        return
    y_test = np.asarray(y_test).ravel()
    pred_test = np.asarray(pred_test).ravel()
    labels_te = np.asarray(labels_te).ravel()
    res = y_test - pred_test

    mask0 = labels_te == 0
    mask1 = labels_te == 1

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    c0 = "#1f77b4"  # Blue for Regime 0
    c1 = "#ff7f0e"  # Orange for Regime 1

    # 1. Regime 0 True vs Pred
    ax = axes[0, 0]
    if mask0.any():
        ax.scatter(y_test[mask0], pred_test[mask0], s=10, alpha=0.5, color=c0, label=f"Regime 0 (N={mask0.sum()})")
        r2_0 = r2_score(y_test[mask0], pred_test[mask0]) if mask0.sum() > 1 else float("nan")
        rmse_0 = root_mean_squared_error(y_test[mask0], pred_test[mask0])
        ax.plot([y_test[mask0].min(), y_test[mask0].max()], [y_test[mask0].min(), y_test[mask0].max()], "k--", lw=1.5)
        ax.set_title(f"Regime 0: True vs Pred\nR² = {r2_0:.4f}, RMSE = {rmse_0:.4f}")
    else:
        ax.set_title("Regime 0 (No Samples)")
    ax.set_xlabel("True Soil Moisture")
    ax.set_ylabel("Predicted Soil Moisture")
    ax.grid(True, linestyle="--", alpha=0.6)

    # 2. Regime 1 True vs Pred
    ax = axes[0, 1]
    if mask1.any():
        ax.scatter(y_test[mask1], pred_test[mask1], s=10, alpha=0.5, color=c1, label=f"Regime 1 (N={mask1.sum()})")
        r2_1 = r2_score(y_test[mask1], pred_test[mask1]) if mask1.sum() > 1 else float("nan")
        rmse_1 = root_mean_squared_error(y_test[mask1], pred_test[mask1])
        ax.plot([y_test[mask1].min(), y_test[mask1].max()], [y_test[mask1].min(), y_test[mask1].max()], "k--", lw=1.5)
        ax.set_title(f"Regime 1: True vs Pred\nR² = {r2_1:.4f}, RMSE = {rmse_1:.4f}")
    else:
        ax.set_title("Regime 1 (No Samples)")
    ax.set_xlabel("True Soil Moisture")
    ax.set_ylabel("Predicted Soil Moisture")
    ax.grid(True, linestyle="--", alpha=0.6)

    # 3. Residual Distribution Overlay
    ax = axes[0, 2]
    if mask0.any():
        ax.hist(res[mask0], bins=30, alpha=0.5, color=c0, density=True, label=f"Regime 0 (bias={np.mean(res[mask0]):+.4f})")
    if mask1.any():
        ax.hist(res[mask1], bins=30, alpha=0.5, color=c1, density=True, label=f"Regime 1 (bias={np.mean(res[mask1]):+.4f})")
    ax.axvline(0, color="k", linestyle="--", lw=1.5)
    ax.set_xlabel("Residual (true - pred)")
    ax.set_ylabel("Density")
    ax.set_title("Residual Distributions (Regime 0 vs 1)")
    ax.legend(fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.6)

    # 4. Regime 0 Residual vs True
    ax = axes[1, 0]
    if mask0.any():
        ax.scatter(y_test[mask0], res[mask0], s=10, alpha=0.5, color=c0)
    ax.axhline(0, color="k", linestyle="--", lw=1.5)
    ax.set_xlabel("True Soil Moisture")
    ax.set_ylabel("Residual (true - pred)")
    ax.set_title("Regime 0 Residuals vs Target")
    ax.grid(True, linestyle="--", alpha=0.6)

    # 5. Regime 1 Residual vs True
    ax = axes[1, 1]
    if mask1.any():
        ax.scatter(y_test[mask1], res[mask1], s=10, alpha=0.5, color=c1)
    ax.axhline(0, color="k", linestyle="--", lw=1.5)
    ax.set_xlabel("True Soil Moisture")
    ax.set_ylabel("Residual (true - pred)")
    ax.set_title("Regime 1 Residuals vs Target")
    ax.grid(True, linestyle="--", alpha=0.6)

    # 6. Residual Boxplots by Year & Regime
    ax = axes[1, 2]
    years = sorted(test_df["year"].unique())
    box_data = []
    box_labels = []
    box_colors = []
    for yr in years:
        mask_yr = (test_df["year"] == yr).values
        m0_yr = mask0 & mask_yr
        m1_yr = mask1 & mask_yr
        if m0_yr.any():
            box_data.append(res[m0_yr])
            box_labels.append(f"{int(yr)}\nR0")
            box_colors.append(c0)
        if m1_yr.any():
            box_data.append(res[m1_yr])
            box_labels.append(f"{int(yr)}\nR1")
            box_colors.append(c1)

    if box_data:
        bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True)
        for patch, color in zip(bp["boxes"], box_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
    ax.axhline(0, color="k", linestyle="--", lw=1.5)
    ax.set_ylabel("Residual (true - pred)")
    ax.set_title("Residuals by Year × Regime")
    ax.grid(True, linestyle="--", alpha=0.6)

    plt.suptitle(f"Per-Regime Diagnostics: {name}", fontsize=16, fontweight="bold", y=0.98)
    plt.tight_layout()

    clean_name = (
        name.lower()
        .replace(" ", "_")
        .replace(":", "")
        .replace(".", "")
        .replace("=", "")
        .replace("(", "")
        .replace(")", "")
        .replace("/", "_")
    )
    fn = f"regime_diagnostics_{clean_name}.png"
    plt.savefig(out_dir / fn, dpi=150)
    plt.close()


def plot_yearly_performance_linechart(metrics_by_year_df, out_dir):
    fig, ax = plt.subplots(figsize=(12, 7))
    models = metrics_by_year_df["Model Name"].unique()
    colors = cm.tab20(np.linspace(0, 1, max(len(models), 1)))
    markers = ["o", "s", "D", "^", "v", "<", ">", "p", "*", "h", "x", "P", "X"]

    for idx, model_name in enumerate(models):
        df_model = metrics_by_year_df[metrics_by_year_df["Model Name"] == model_name].sort_values("Year")
        ax.plot(
            df_model["Year"].values,
            df_model["R2"].values,
            label=model_name,
            color=colors[idx % len(colors)],
            marker=markers[idx % len(markers)],
            linewidth=2,
        )

    ax.set_xticks([2023, 2024, 2025])
    ax.set_xticklabels(["2023", "2024", "2025"])
    ax.set_xlabel("Test Year", fontweight="bold")
    ax.set_ylabel("$R^2$ Score", fontweight="bold")
    ax.set_title("Model R2 Performance Over Test Years (2023-2025)", fontsize=14, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.savefig(out_dir / "r2_performance_over_years.png", dpi=150)
    plt.close()


# New SOTA 1.5 HPs (from derived_8.2-hyperparameters-1.5)
XGB_REG_PARAMS = {
    "objective": "reg:squarederror",
    "max_depth": 9,
    "min_child_weight": 8,
    "reg_lambda": 0.75,
    "reg_alpha": 0.03,
    "subsample": 0.9,
    "colsample_bytree": 0.8,
    "n_estimators": 2500,
    "learning_rate": 0.005,
    "random_state": SEED,
    "n_jobs": 1,
    "device": XGB_DEVICE,
}

XGB_CLF_PARAMS = {
    "objective": "binary:logistic",
    "max_depth": 9,
    "min_child_weight": 8,
    "reg_lambda": 0.75,
    "reg_alpha": 0.03,
    "subsample": 0.9,
    "colsample_bytree": 0.8,
    "n_estimators": 2500,
    "learning_rate": 0.005,
    "random_state": SEED,
    "n_jobs": 1,
    "device": XGB_DEVICE,
}

N_EST = XGB_REG_PARAMS["n_estimators"]
COLS_DYNAMIC = ["SMAP_sm_pm_interp_lag1", "G_API", "LST_modis"]
print("Helpers ready. XGB_REG_PARAMS n_estimators =", N_EST)


Helpers ready. XGB_REG_PARAMS n_estimators = 2500


# Section 5: Model matrix and parallel evaluation

Define the 18-model matrix (2 baselines + 4 K=2 strategies × 4 feature arms). Each config is trained in a thread-pool worker with CRC32-keyed disk cache, load-retry, and per-future exception handling so one failure cannot abort the sweep. Specialists within a model train sequentially inside that worker.


In [ ]:
STRATEGIES = [
    "trained_gating_k2",
    "Univariate_G_API_k2",
    "Clustering_Dynamic_k2",
    "Seasonal_Binary_k2",
]
ARMS = ["spec_old", "spec_new", "global_v3", "global_c1"]
ARM_LABEL = {
    "spec_old": "Spec-old",
    "spec_new": "Spec-new",
    "global_v3": "Global-V3",
    "global_c1": "Global-c1",
}
STRAT_LABEL = {
    "trained_gating_k2": "Trained Gating K=2",
    "Univariate_G_API_k2": "Univariate G_API K=2",
    "Clustering_Dynamic_k2": "Clustering Dynamic K=2",
    "Seasonal_Binary_k2": "Seasonal Binary K=2",
}

MODELS_CONFIG = [
    {"id": 1, "name": "Model 1: Baseline V3", "type": "baseline", "arm": "global_v3", "strat": None},
    {"id": 2, "name": "Model 2: Baseline c1", "type": "baseline", "arm": "global_c1", "strat": None},
]
mid = 3
for strat in STRATEGIES:
    for arm in ARMS:
        MODELS_CONFIG.append({
            "id": mid,
            "name": f"Model {mid}: {STRAT_LABEL[strat]} ({ARM_LABEL[arm]})",
            "type": "moe",
            "strat": strat,
            "arm": arm,
            "K": 2,
        })
        mid += 1

assert len(MODELS_CONFIG) == 18, len(MODELS_CONFIG)
print("Model matrix:")
for c in MODELS_CONFIG:
    print(f"  {c['id']:2d}. {c['name']}")


def resolve_specialist_features(strat, arm, cluster_id):
    if arm == "global_v3":
        return list(FEATURE_SET_V3)
    if arm == "global_c1":
        return list(FEATURE_SET_C1)

    if strat == "trained_gating_k2":
        regime = "dry" if cluster_id == 0 else "wet"
        src = REGIME_OLD if arm == "spec_old" else REGIME_NEW
        return list(src[regime])

    # unsupervised cluster strategies
    src = CLUSTER_OLD if arm == "spec_old" else CLUSTER_NEW
    key = str(cluster_id)
    if strat not in src or key not in src[strat]:
        raise KeyError(f"Missing features for {strat}/{key} arm={arm}")
    return list(src[strat][key])


def get_route_labels(strat, K=2):
    if strat == "trained_gating_k2":
        y_gate_trval = np.where(y_trval < T_BINARY, 0, 1)
        y_gate_te_true = np.where(y_te < T_BINARY, 0, 1)
        return {
            "mode": "learned",
            "y_gate_trval": y_gate_trval,
            "y_gate_te_true": y_gate_te_true,
        }

    if strat == "Univariate_G_API_k2":
        binner = QuantileBinner(K)
        binner.fit(train_df["G_API"])
        return {
            "mode": "fixed",
            "labels_trval": binner.predict(trainval_df["G_API"]),
            "labels_te": binner.predict(test_df["G_API"]),
        }

    if strat == "Clustering_Dynamic_k2":
        clusterer = KMeansClusterer(COLS_DYNAMIC, K)
        clusterer.fit(train_df)
        return {
            "mode": "fixed",
            "labels_trval": clusterer.predict(trainval_df),
            "labels_te": clusterer.predict(test_df),
        }

    if strat == "Seasonal_Binary_k2":
        cond_trval = [
            trainval_df["month"].isin([5, 6, 7, 8, 9, 10]),
            trainval_df["month"].isin([11, 12, 1, 2, 3, 4]),
        ]
        cond_te = [
            test_df["month"].isin([5, 6, 7, 8, 9, 10]),
            test_df["month"].isin([11, 12, 1, 2, 3, 4]),
        ]
        return {
            "mode": "fixed",
            "labels_trval": np.select(cond_trval, [0, 1], default=0),
            "labels_te": np.select(cond_te, [0, 1], default=0),
        }

    raise ValueError(f"Unknown strat: {strat}")


models_dir = out_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)
_save_lock = threading.Lock()
_print_lock = threading.Lock()


def config_crc32(config: dict) -> str:
    payload = {
        "id": config["id"],
        "name": config["name"],
        "type": config["type"],
        "arm": config.get("arm"),
        "strat": config.get("strat"),
        "K": config.get("K", 1),
        "reg_params": XGB_REG_PARAMS,
        "clf_params": XGB_CLF_PARAMS,
    }
    if config["type"] == "moe":
        payload["specialist_features"] = {
            str(c): sorted(resolve_specialist_features(config["strat"], config["arm"], c))
            for c in range(config["K"])
        }
    blob = json.dumps(payload, sort_keys=True).encode("utf-8")
    return f"{zlib.crc32(blob):08x}"


def model_paths(config):
    crc = config_crc32(config)
    mid = config["id"]
    return {
        "crc": crc,
        "pred_cache": models_dir / f"model_{mid}_{crc}_preds.npy",
        "label_cache": models_dir / f"model_{mid}_{crc}_labels_te.npy",
        "curve_cache": models_dir / f"model_{mid}_{crc}_curve.npy",
        "meta_path": models_dir / f"model_{mid}_{crc}_meta.json",
    }


def save_booster(model, path: Path):
    tmp = path.parent / f".{path.name}.tmp"
    model.save_model(tmp)
    tmp.replace(path)


def load_booster(path: Path, is_classifier=False):
    model = XGBClassifier() if is_classifier else XGBRegressor()
    model.load_model(path)
    return model


def combine_rmse_curves(spec_rmse_curves, spec_N_tests, n_steps=N_EST):
    total_N = sum(spec_N_tests)
    if total_N == 0:
        return [0.0] * n_steps
    combined = np.zeros(n_steps, dtype=float)
    for curve, N_c in zip(spec_rmse_curves, spec_N_tests):
        if len(curve) != n_steps:
            full = np.zeros(n_steps, dtype=float)
            full[: len(curve)] = curve
            if len(curve) > 0:
                full[len(curve) :] = curve[-1]
            curve = full
        combined += (N_c / total_N) * (np.asarray(curve) ** 2)
    return np.sqrt(combined).tolist()


def train_one(config):
    paths = model_paths(config)
    crc = paths["crc"]
    model_id = config["id"]
    model_name = config["name"]
    pred_cache = paths["pred_cache"]
    label_cache = paths["label_cache"]
    curve_cache = paths["curve_cache"]
    meta_path = paths["meta_path"]

    if pred_cache.exists() and curve_cache.exists() and meta_path.exists():
        try:
            with open(meta_path, "r") as f:
                meta = json.load(f)
            preds = np.load(pred_cache)
            curve = np.load(curve_cache).tolist()
            labels_te = np.load(label_cache) if label_cache.exists() else None
            with _print_lock:
                print(f"[{model_id}] Loading {model_name} (crc={crc}) from cache...")
            return model_id, {
                "preds": preds,
                "labels_te": labels_te,
                "rmse_curve": curve,
                "train_time_s": meta["train_time_s"],
                "gating": meta.get("gating"),
                "name": meta["name"],
                "crc": crc,
                "loaded": True,
            }
        except Exception as e:
            with _print_lock:
                print(f"[{model_id}] Cache read failed ({e}), re-training...")

    with _print_lock:
        print(f"[{model_id}] Training {model_name} (crc={crc})...")

    t0 = time.perf_counter()
    pred_test = np.zeros(len(test_df), dtype=float)
    gating_info = None
    labels_te = None

    if config["type"] == "baseline":
        feats = (
            FEATURE_SET_V3
            if config["arm"] == "global_v3"
            else FEATURE_SET_C1
        )
        model = XGBRegressor(**XGB_REG_PARAMS)
        model.fit(
            trainval_df[feats],
            y_trval,
            eval_set=[(test_df[feats], y_te)],
            verbose=False,
        )
        pred_test = np.asarray(model.predict(test_df[feats])).ravel()
        rmse_curve = list(model.evals_result()["validation_0"]["rmse"])
        with _save_lock:
            save_booster(model, models_dir / f"model_{model_id}_{crc}_reg.json")

    else:
        strat = config["strat"]
        arm = config["arm"]
        K = config["K"]
        route = get_route_labels(strat, K)

        if route["mode"] == "learned":
            y_gate_trval = route["y_gate_trval"]
            y_gate_te_true = route["y_gate_te_true"]
            gate_feats = FEATURE_SET_V3
            gating_clf = XGBClassifier(**XGB_CLF_PARAMS)
            gating_clf.fit(trainval_df[gate_feats], y_gate_trval, verbose=False)
            pred_gate_te = np.asarray(gating_clf.predict(test_df[gate_feats])).ravel()
            labels_trval = y_gate_trval
            labels_te = pred_gate_te

            acc = accuracy_score(y_gate_te_true, pred_gate_te)
            prec, rec, f1, _ = precision_recall_fscore_support(
                y_gate_te_true, pred_gate_te, average="macro", zero_division=0
            )
            gating_info = {
                "Accuracy": float(acc),
                "Precision": float(prec),
                "Recall": float(rec),
                "F1": float(f1),
                "K": K,
            }
            with _save_lock:
                save_booster(
                    gating_clf,
                    models_dir / f"model_{model_id}_{crc}_gating.json",
                )
        else:
            labels_trval = np.asarray(route["labels_trval"]).ravel()
            labels_te = np.asarray(route["labels_te"]).ravel()

        spec_rmse_curves = []
        spec_N_tests = []
        for c in range(K):
            feats = resolve_specialist_features(strat, arm, c)
            mask_trval = labels_trval == c
            mask_te_sub = labels_te == c
            X_trval_sub = trainval_df.loc[mask_trval, feats]
            y_trval_sub = y_trval[mask_trval]
            X_te_sub = test_df.loc[mask_te_sub, feats]
            y_te_sub = y_te[mask_te_sub]

            specialist = XGBRegressor(**XGB_REG_PARAMS)
            if len(X_te_sub) > 0 and len(X_trval_sub) > 0:
                specialist.fit(
                    X_trval_sub,
                    y_trval_sub,
                    eval_set=[(X_te_sub, y_te_sub)],
                    verbose=False,
                )
                curve = list(specialist.evals_result()["validation_0"]["rmse"])
            elif len(X_trval_sub) > 0:
                specialist.fit(X_trval_sub, y_trval_sub, verbose=False)
                curve = [0.0] * N_EST
            else:
                curve = [0.0] * N_EST
                if mask_te_sub.any():
                    pred_test[mask_te_sub] = float(np.mean(y_trval))
                spec_rmse_curves.append(curve)
                spec_N_tests.append(int(mask_te_sub.sum()))
                continue

            with _save_lock:
                save_booster(
                    specialist,
                    models_dir / f"model_{model_id}_{crc}_spec{c}.json",
                )
            if mask_te_sub.any():
                pred_test[mask_te_sub] = np.asarray(
                    specialist.predict(X_te_sub)
                ).ravel()
            spec_rmse_curves.append(curve)
            spec_N_tests.append(int(mask_te_sub.sum()))

        rmse_curve = combine_rmse_curves(spec_rmse_curves, spec_N_tests)

    train_time = time.perf_counter() - t0
    meta = {
        "train_time_s": train_time,
        "name": model_name,
        "config_crc32": crc,
        "config_id": model_id,
        "gating": gating_info,
        "arm": config.get("arm"),
        "strat": config.get("strat"),
        "type": config["type"],
    }
    with _save_lock:
        np.save(pred_cache, pred_test)
        if labels_te is not None:
            np.save(label_cache, labels_te)
        np.save(curve_cache, np.asarray(rmse_curve, dtype=float))
        tmp_meta = meta_path.parent / f".{meta_path.stem}.writing.json"
        with open(tmp_meta, "w") as f:
            json.dump(meta, f, indent=2)
        tmp_meta.replace(meta_path)

    with _print_lock:
        print(f"[{model_id}] Done in {train_time:.2f}s (crc={crc})")

    return model_id, {
        "preds": pred_test,
        "labels_te": labels_te,
        "rmse_curve": rmse_curve,
        "train_time_s": train_time,
        "gating": gating_info,
        "name": model_name,
        "crc": crc,
        "loaded": False,
    }


print(f"\nStarting parallel training for {len(MODELS_CONFIG)} models "
      f"with {PARALLEL_WORKERS} workers...\n")
wall_t0 = time.perf_counter()
results_by_id = {}
failed = []
completed = 0

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
    futures = {executor.submit(train_one, cfg): cfg for cfg in MODELS_CONFIG}
    for fut in as_completed(futures):
        cfg = futures[fut]
        mid = cfg["id"]
        try:
            mid, result = fut.result()
            results_by_id[mid] = result
        except Exception as e:
            failed.append({
                "id": mid,
                "name": cfg.get("name"),
                "error": f"{type(e).__name__}: {e}",
            })
            with _print_lock:
                print(f"[{mid}] FAILED: {type(e).__name__}: {e}")
        completed += 1
        if completed % 3 == 0 or completed == len(MODELS_CONFIG):
            elapsed = time.perf_counter() - wall_t0
            rate = completed / elapsed if elapsed > 0 else 0
            remaining = (len(MODELS_CONFIG) - completed) / rate if rate > 0 else float("nan")
            n_loaded = sum(1 for r in results_by_id.values() if r.get("loaded"))
            with _print_lock:
                print(
                    f"  Progress: {completed}/{len(MODELS_CONFIG)} "
                    f"({elapsed:.1f}s elapsed, ~{remaining:.1f}s remaining, "
                    f"{len(failed)} failed, {n_loaded} loaded from cache)"
                )

wall_t1 = time.perf_counter()
print(
    f"\nProcessed {completed} configs in {wall_t1 - wall_t0:.1f}s wall time "
    f"({len(results_by_id)} ok, {len(failed)} failed)."
)

if failed:
    failed_df = pd.DataFrame(failed)
    failed_path = out_dir / "failed_configs.csv"
    failed_df.to_csv(failed_path, index=False)
    print(f"WARNING: {len(failed)} configs failed. See {failed_path}")
else:
    failed_path = out_dir / "failed_configs.csv"
    if failed_path.exists():
        failed_path.unlink()

# Assemble metrics
overall_metrics = []
per_regime_metrics = []
yearly_metrics = []
gating_perf_metrics = []
all_rmse_curves = {}

for config in MODELS_CONFIG:
    mid = config["id"]
    mname = config["name"]
    if mid not in results_by_id:
        print(f"[SKIP] No results for {mname}")
        continue
    result = results_by_id[mid]
    pred_test = result["preds"]
    labels_te = result.get("labels_te")
    all_rmse_curves[mname] = result["rmse_curve"]

    metrics_ov = compute_metrics(y_te, pred_test)
    metrics_ov["Model Name"] = mname
    metrics_ov["Model ID"] = mid
    metrics_ov["Arm"] = config.get("arm")
    metrics_ov["Strategy"] = config.get("strat")
    metrics_ov["Train Time (s)"] = result["train_time_s"]
    overall_metrics.append(metrics_ov)
    print(f"{mname}: R2 = {metrics_ov['R2']:.4f}")

    if labels_te is not None:
        mask0 = labels_te == 0
        mask1 = labels_te == 1
        m_r0 = compute_metrics(y_te[mask0], pred_test[mask0]) if mask0.any() else {}
        m_r1 = compute_metrics(y_te[mask1], pred_test[mask1]) if mask1.any() else {}
        per_regime_metrics.append({
            "Model ID": mid,
            "Model Name": mname,
            "Strategy": config.get("strat"),
            "Arm": config.get("arm"),
            "N_R0": int(mask0.sum()),
            "R2_R0": m_r0.get("R2", float("nan")),
            "RMSE_R0": m_r0.get("RMSE", float("nan")),
            "Bias_R0": m_r0.get("Bias", float("nan")),
            "MAE_R0": m_r0.get("MAE", float("nan")),
            "N_R1": int(mask1.sum()),
            "R2_R1": m_r1.get("R2", float("nan")),
            "RMSE_R1": m_r1.get("RMSE", float("nan")),
            "Bias_R1": m_r1.get("Bias", float("nan")),
            "MAE_R1": m_r1.get("MAE", float("nan")),
        })

    for yr in test_years:
        mask_yr = (test_df["year"] == yr).values
        metrics_yr = compute_metrics(y_te[mask_yr], pred_test[mask_yr])
        metrics_yr["Model Name"] = mname
        metrics_yr["Model ID"] = mid
        metrics_yr["Year"] = int(yr)
        yearly_metrics.append(metrics_yr)

    if result.get("gating"):
        g = dict(result["gating"])
        g["Model Name"] = mname
        gating_perf_metrics.append(g)

    plot_diagnostics(mname, y_te, pred_test, test_df, out_dir)
    if labels_te is not None:
        plot_per_regime_diagnostics(mname, y_te, pred_test, labels_te, test_df, out_dir)

# Save curves
rmse_curves_df = pd.DataFrame(all_rmse_curves)
rmse_curves_df.index.name = "Step"
rmse_curves_df.to_csv(out_dir / "all_models_loss_curves.csv")
print(f"Saved loss curves to {out_dir / 'all_models_loss_curves.csv'}")

if failed:
    raise RuntimeError(
        f"{len(failed)} configs failed (see failed_configs.csv). "
        f"{len(results_by_id)} succeeded — re-run to resume."
    )


Model matrix:
   1. Model 1: Baseline V3
   2. Model 2: Baseline c1
   3. Model 3: Trained Gating K=2 (Spec-old)
   4. Model 4: Trained Gating K=2 (Spec-new)
   5. Model 5: Trained Gating K=2 (Global-V3)
   6. Model 6: Trained Gating K=2 (Global-c1)
   7. Model 7: Univariate G_API K=2 (Spec-old)
   8. Model 8: Univariate G_API K=2 (Spec-new)
   9. Model 9: Univariate G_API K=2 (Global-V3)
  10. Model 10: Univariate G_API K=2 (Global-c1)
  11. Model 11: Clustering Dynamic K=2 (Spec-old)
  12. Model 12: Clustering Dynamic K=2 (Spec-new)
  13. Model 13: Clustering Dynamic K=2 (Global-V3)
  14. Model 14: Clustering Dynamic K=2 (Global-c1)
  15. Model 15: Seasonal Binary K=2 (Spec-old)
  16. Model 16: Seasonal Binary K=2 (Spec-new)
  17. Model 17: Seasonal Binary K=2 (Global-V3)
  18. Model 18: Seasonal Binary K=2 (Global-c1)

Starting parallel training for 18 models with 4 workers...

[1] Loading Model 1: Baseline V3 (crc=dae5c5b9) from cache...
[2] Loading Model 2: Baseline c1 (crc=1608a7

[9] Done in 180.28s (crc=0d74a4dd)
[10] Done in 181.42s (crc=18528550)


  Progress: 9/18 (182.7s elapsed, ~182.7s remaining, 0 failed, 8 loaded from cache)
[13] Training Model 13: Clustering Dynamic K=2 (Global-V3) (crc=a31f7e11)...
[14] Training Model 14: Clustering Dynamic K=2 (Global-c1) (crc=615b9d6f)...


[11] Done in 182.89s (crc=f89e96c3)


[15] Training Model 15: Seasonal Binary K=2 (Spec-old) (crc=96d678f9)...


...

Model 15: Seasonal Binary K=2 (Spec-old): R2 = 0.5964


Model 16: Seasonal Binary K=2 (Spec-new): R2 = 0.6190


Model 17: Seasonal Binary K=2 (Global-V3): R2 = 0.6435


Model 18: Seasonal Binary K=2 (Global-c1): R2 = 0.6446


Saved loss curves to C:\Users\pan\Documents\GitHub\MDR-Project\notebooks\experiment\derived_8.2-eval-3.3\all_models_loss_curves.csv


# Section 6: Results tables and figures

Aggregate overall and yearly metrics, write CSVs, plot R² by year, and produce consolidated and strategy-grouped loss-curve figures comparing Spec-old / Spec-new / Global-V3 / Global-c1.


In [ ]:
overall_metrics_df = pd.DataFrame(overall_metrics).sort_values("R2", ascending=False)
print("=== OVERALL METRICS (sorted by R2) ===")
print(
    overall_metrics_df[
        ["Model ID", "Model Name", "Arm", "Strategy", "R2", "RMSE", "ubRMSE", "Bias", "MAE", "Med|Err|", "Pearson", "Train Time (s)"]
    ].to_string(
        index=False,
        formatters={
            "R2": "{:,.4f}".format,
            "RMSE": "{:,.4f}".format,
            "ubRMSE": "{:,.4f}".format,
            "Bias": "{:+,.4f}".format,
            "MAE": "{:,.4f}".format,
            "Med|Err|": "{:,.4f}".format,
            "Pearson": "{:,.4f}".format,
            "Train Time (s)": "{:,.1f}".format,
        },
    )
)
overall_metrics_df.to_csv(out_dir / "metrics_summary.csv", index=False)
print(f"\nSaved overall metrics to: {out_dir / 'metrics_summary.csv'}")

if per_regime_metrics:
    per_regime_df = pd.DataFrame(per_regime_metrics)
    per_regime_df.to_csv(out_dir / "per_regime_metrics_summary.csv", index=False)
    print(f"Saved per-regime metrics summary to: {out_dir / 'per_regime_metrics_summary.csv'}")
    print("\n=== PER-REGIME METRICS SUMMARY ===")
    print(
        per_regime_df[
            ["Model ID", "Model Name", "N_R0", "R2_R0", "RMSE_R0", "Bias_R0", "N_R1", "R2_R1", "RMSE_R1", "Bias_R1"]
        ].to_string(
            index=False,
            formatters={
                "R2_R0": "{:,.4f}".format,
                "RMSE_R0": "{:,.4f}".format,
                "Bias_R0": "{:+,.4f}".format,
                "R2_R1": "{:,.4f}".format,
                "RMSE_R1": "{:,.4f}".format,
                "Bias_R1": "{:+,.4f}".format,
            },
        )
    )

yearly_metrics_df = pd.DataFrame(yearly_metrics)
yearly_metrics_df.to_csv(out_dir / "metrics_by_year.csv", index=False)
print(f"Saved yearly metrics to: {out_dir / 'metrics_by_year.csv'}")

# Yearly R2 pivot
pivot = yearly_metrics_df.pivot_table(index="Model Name", columns="Year", values="R2")
overall_r2 = overall_metrics_df.set_index("Model Name")["R2"]
pivot.insert(0, "Overall", overall_r2)
print("\n=== R2 BY YEAR ===")
print(pivot.round(4).to_string())

plot_yearly_performance_linechart(yearly_metrics_df, out_dir)

if len(gating_perf_metrics) > 0:
    gating_perf_df = pd.DataFrame(gating_perf_metrics)
    print("\n=== GATING ROUTER PERFORMANCE (trained gating only) ===")
    print(gating_perf_df.to_string(index=False))
    gating_perf_df.to_csv(out_dir / "gating_performance_summary.csv", index=False)
else:
    print("No gating metrics collected.")

# Loss curves: consolidated
curves_df = rmse_curves_df
plt.figure(figsize=(14, 8))
colors = cm.tab20(np.linspace(0, 1, len(curves_df.columns)))
for idx, col in enumerate(curves_df.columns):
    plt.plot(curves_df.index, curves_df[col], label=col, color=colors[idx], linewidth=1.2)
plt.xlabel("Training Step (Boosting Iteration)", fontweight="bold")
plt.ylabel("Test RMSE", fontweight="bold")
plt.title("Consolidated Loss Curves (RMSE) on Test Set", fontsize=14, fontweight="bold")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=7)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(out_dir / "loss_curves_consolidated.png", dpi=150)
plt.close()

# Grouped by strategy: 4 arms + baselines
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()
arm_styles = {
    "spec_old": ("#1f77b4", "-"),
    "spec_new": ("#2ca02c", "-"),
    "global_v3": ("#d62728", "--"),
    "global_c1": ("#ff7f0e", "--"),
}
baseline_v3_name = "Model 1: Baseline V3"
baseline_c1_name = "Model 2: Baseline c1"

for idx, strat in enumerate(STRATEGIES):
    ax = axes[idx]
    if baseline_v3_name in curves_df.columns:
        ax.plot(curves_df.index, curves_df[baseline_v3_name], "k-", alpha=0.5, label="Baseline V3", linewidth=1.5)
    if baseline_c1_name in curves_df.columns:
        ax.plot(curves_df.index, curves_df[baseline_c1_name], "k--", alpha=0.5, label="Baseline c1", linewidth=1.5)
    for arm in ARMS:
        matches = [
            c["name"]
            for c in MODELS_CONFIG
            if c.get("strat") == strat and c.get("arm") == arm
        ]
        if not matches or matches[0] not in curves_df.columns:
            continue
        color, ls = arm_styles[arm]
        ax.plot(
            curves_df.index,
            curves_df[matches[0]],
            color=color,
            linestyle=ls,
            label=ARM_LABEL[arm],
            linewidth=2,
        )
    ax.set_title(STRAT_LABEL[strat], fontsize=12, fontweight="bold")
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Test RMSE")
    ax.legend(fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.5)

plt.suptitle(
    "Loss Curves by Strategy: Spec-old vs Spec-new vs Global-V3 vs Global-c1",
    fontsize=14,
    fontweight="bold",
    y=0.995,
)
plt.tight_layout()
plt.savefig(out_dir / "loss_curves_grouped.png", dpi=150)
plt.close()

# Compact ablation table: best R2 per (strategy, arm)
ablation_rows = []
for c in MODELS_CONFIG:
    if c["type"] != "moe":
        continue
    row = overall_metrics_df[overall_metrics_df["Model ID"] == c["id"]]
    if row.empty:
        continue
    ablation_rows.append({
        "Strategy": STRAT_LABEL[c["strat"]],
        "Arm": ARM_LABEL[c["arm"]],
        "R2": float(row["R2"].iloc[0]),
        "RMSE": float(row["RMSE"].iloc[0]),
    })
ablation_df = pd.DataFrame(ablation_rows)
if not ablation_df.empty:
    pivot_abl = ablation_df.pivot(index="Strategy", columns="Arm", values="R2")
    col_order = [ARM_LABEL[a] for a in ARMS if ARM_LABEL[a] in pivot_abl.columns]
    pivot_abl = pivot_abl[col_order]
    print("\n=== ABLATION: R2 by Strategy × Feature Arm ===")
    print(pivot_abl.round(4).to_string())
    pivot_abl.to_csv(out_dir / "ablation_r2_strategy_x_arm.csv")

# Consolidated Per-Regime Comparison Visualizations
if per_regime_metrics:
    pr_df = pd.DataFrame(per_regime_metrics)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    x = np.arange(len(pr_df))
    width = 0.35

    # R2 comparison
    ax = axes[0]
    ax.bar(x - width/2, pr_df["R2_R0"], width, label="Regime 0", color="#1f77b4", alpha=0.85)
    ax.bar(x + width/2, pr_df["R2_R1"], width, label="Regime 1", color="#ff7f0e", alpha=0.85)
    ax.set_ylabel("$R^2$ Score", fontweight="bold")
    ax.set_title("Per-Regime $R^2$ Score across 2-Regime Models", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([f"M{mid}" for mid in pr_df["Model ID"]], rotation=45, ha="right")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)

    # RMSE comparison
    ax = axes[1]
    ax.bar(x - width/2, pr_df["RMSE_R0"], width, label="Regime 0", color="#1f77b4", alpha=0.85)
    ax.bar(x + width/2, pr_df["RMSE_R1"], width, label="Regime 1", color="#ff7f0e", alpha=0.85)
    ax.set_ylabel("RMSE", fontweight="bold")
    ax.set_title("Per-Regime RMSE across 2-Regime Models", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([f"M{mid}" for mid in pr_df["Model ID"]], rotation=45, ha="right")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)

    plt.suptitle("Per-Regime Performance Comparison (Regime 0 vs Regime 1)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_dir / "per_regime_r2_rmse_comparison.png", dpi=150)
    plt.close()

    # 2. Per-Regime Residual Boxplot Summary across Models
    fig, ax = plt.subplots(figsize=(16, 7))
    box_data = []
    box_labels = []
    box_colors = []

    for _, row in pr_df.iterrows():
        mid = int(row["Model ID"])
        result = results_by_id[mid]
        preds = result["preds"]
        labels_te = result.get("labels_te")
        if labels_te is None:
            continue
        res = y_te - preds
        mask0 = labels_te == 0
        mask1 = labels_te == 1
        if mask0.any():
            box_data.append(res[mask0])
            box_labels.append(f"M{mid}\nR0")
            box_colors.append("#1f77b4")
        if mask1.any():
            box_data.append(res[mask1])
            box_labels.append(f"M{mid}\nR1")
            box_colors.append("#ff7f0e")

    if box_data:
        bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True)
        for patch, color in zip(bp["boxes"], box_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
    ax.axhline(0, color="k", linestyle="--", lw=1.5)
    ax.set_ylabel("Residual (true - pred)", fontweight="bold")
    ax.set_title("Residual Distributions by Model × Regime (Regime 0 vs Regime 1)", fontsize=14, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.tight_layout()
    plt.savefig(out_dir / "per_regime_residuals_boxplot.png", dpi=150)
    plt.close()

print("\nDone. Key outputs:")
for p in [
    "metrics_summary.csv",
    "per_regime_metrics_summary.csv",
    "metrics_by_year.csv",
    "gating_performance_summary.csv",
    "all_models_loss_curves.csv",
    "ablation_r2_strategy_x_arm.csv",
    "r2_performance_over_years.png",
    "loss_curves_consolidated.png",
    "loss_curves_grouped.png",
    "per_regime_r2_rmse_comparison.png",
    "per_regime_residuals_boxplot.png",
]:
    fp = out_dir / p
    print(f"  {'OK' if fp.exists() else '— '} {p}")


=== OVERALL METRICS (sorted by R2) ===
 Model ID                                   Model Name       Arm              Strategy     R2   RMSE ubRMSE    Bias    MAE Med|Err| Pearson Train Time (s)
       14 Model 14: Clustering Dynamic K=2 (Global-c1) global_c1 Clustering_Dynamic_k2 0.6639 0.0610 0.0588 -0.0163 0.0447   0.0330  0.8306          186.5
        2                         Model 2: Baseline c1 global_c1                   NaN 0.6637 0.0611 0.0593 -0.0146 0.0447   0.0337  0.8270          173.0
        1                         Model 1: Baseline V3 global_v3                   NaN 0.6589 0.0615 0.0577 -0.0212 0.0459   0.0349  0.8396          167.7
       18    Model 18: Seasonal Binary K=2 (Global-c1) global_c1    Seasonal_Binary_k2 0.6446 0.0628 0.0609 -0.0153 0.0454   0.0334  0.8176           85.1
       10   Model 10: Univariate G_API K=2 (Global-c1) global_c1   Univariate_G_API_k2 0.6445 0.0628 0.0613 -0.0137 0.0454   0.0339  0.8146          181.4
       17    Model 17: Seasonal


=== GATING ROUTER PERFORMANCE (trained gating only) ===
 Accuracy  Precision   Recall       F1  K                              Model Name
  0.87441   0.866308 0.866394 0.866351  2  Model 3: Trained Gating K=2 (Spec-old)
  0.87441   0.866308 0.866394 0.866351  2  Model 4: Trained Gating K=2 (Spec-new)
  0.87441   0.866308 0.866394 0.866351  2 Model 5: Trained Gating K=2 (Global-V3)
  0.87441   0.866308 0.866394 0.866351  2 Model 6: Trained Gating K=2 (Global-c1)



=== ABLATION: R2 by Strategy × Feature Arm ===
Arm                     Spec-old  Spec-new  Global-V3  Global-c1
Strategy                                                        
Clustering Dynamic K=2    0.6263    0.6271     0.6102     0.6639
Seasonal Binary K=2       0.5964    0.6190     0.6435     0.6446
Trained Gating K=2        0.5782    0.5856     0.5837     0.6174
Univariate G_API K=2      0.5381    0.5363     0.6383     0.6445

Done. Key outputs:
  OK metrics_summary.csv
  OK metrics_by_year.csv
  OK gating_performance_summary.csv
  OK all_models_loss_curves.csv
  OK ablation_r2_strategy_x_arm.csv
  OK r2_performance_over_years.png
  OK loss_curves_consolidated.png
  OK loss_curves_grouped.png
